In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args
import matplotlib.pyplot as plt

# 1. 加载数据
print("正在加载数据...")
# 修正文件路径，使用相同的平滑数据
df_original = pd.read_excel('../../data/pinghua_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_original, y_original, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# 6. 定义搜索空间 - 针对MLP的参数
dimensions = [
    Integer(50, 200, name='hidden1'),  # 第一层神经元数量
    Integer(10, 100, name='hidden2'),  # 第二层神经元数量
    Categorical(['relu', 'tanh'], name='activation'),  # 激活函数
    Real(0.00001, 0.01, 'log-uniform', name='alpha'),  # L2正则化项
    Real(0.0001, 0.1, 'log-uniform', name='learning_rate_init'),  # 初始学习率
    Categorical(['constant', 'adaptive'], name='learning_rate'),  # 学习率类型
    Categorical([True, False], name='early_stopping'),  # 是否启用早停
    Integer(5, 20, name='n_iter_no_change')  # 早停等待迭代次数
]

# 7. 定义目标函数 - 使用测试集R²作为优化目标
@use_named_args(dimensions)
def objective(hidden1, hidden2, activation, alpha, learning_rate_init, 
              learning_rate, early_stopping, n_iter_no_change):
    # 组合参数
    params = {
        'hidden_layer_sizes': (hidden1, hidden2),
        'activation': activation,
        'solver': 'adam',
        'alpha': alpha,
        'batch_size': 'auto',
        'learning_rate': learning_rate,
        'learning_rate_init': learning_rate_init,
        'max_iter': 1000,
        'early_stopping': early_stopping,
        'n_iter_no_change': n_iter_no_change,
        'validation_fraction': 0.1,
        'tol': 0.0001,
        'random_state': 42
    }
    
    # 使用参数创建MLP模型
    base_model = MLPRegressor(**params)
    model = MultiOutputRegressor(base_model)
    
    try:
        # 在训练集上训练模型
        model.fit(X_train_scaled, y_train)
        
        # 在测试集上预测
        y_pred = model.predict(X_test_scaled)
        
        # 计算测试集上每个目标变量的R²
        r2_scores = []
        for idx in range(len(target_vars)):
            r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
            r2_scores.append(r2)
        
        # 找出最小的R²值和平均R²值
        min_r2 = min(r2_scores)
        mean_r2 = np.mean(r2_scores)
        
        # 打印当前迭代的参数和结果
        print(f"测试参数: hidden1={hidden1}, hidden2={hidden2}, activation={activation}, "
              f"alpha={alpha:.6f}, lr_init={learning_rate_init:.6f}, lr={learning_rate}, "
              f"early_stopping={early_stopping}, n_iter_no_change={n_iter_no_change}")
        
        for idx, name in enumerate(target_vars):
            print(f"  {name} R²: {r2_scores[idx]:.4f}")
        
        print(f"  最小R²: {min_r2:.4f}, 平均R²: {mean_r2:.4f}")
        
        # 返回负最小R²，我们希望最差的表现也要好
        return -min_r2
    
    except Exception as e:
        print(f"错误: {e}")
        return 1000  # 返回一个很大的值表示失败

# 8. 执行贝叶斯优化
print("\n开始基于最小R²的贝叶斯参数优化...")
result = gp_minimize(
    objective,
    dimensions=dimensions,
    n_calls=30,  # 迭代次数
    n_random_starts=10,  # 随机搜索起点数
    random_state=42,
    verbose=True,
    n_jobs=-1
)

# 9. 提取最佳参数
best_hidden1 = result.x[0]
best_hidden2 = result.x[1]
best_activation = result.x[2]
best_alpha = result.x[3]
best_learning_rate_init = result.x[4]
best_learning_rate = result.x[5]
best_early_stopping = result.x[6]
best_n_iter_no_change = result.x[7]






# 10. 构建最佳参数字典
best_params = {
    'hidden_layer_sizes': (best_hidden1, best_hidden2),
    'activation': best_activation,
    'alpha': best_alpha,
    'learning_rate_init': best_learning_rate_init,
    'learning_rate': best_learning_rate,
    'early_stopping': best_early_stopping,
    'n_iter_no_change': best_n_iter_no_change,
    'max_iter': 1000,
    'solver': 'adam',
    'random_state': 42,
    'validation_fraction': 0.1,
    'tol': 0.0001,
    'batch_size': 'auto'
}

print("\n最优参数:")
for param, value in best_params.items():
    print(f"{param}: {value}")

# 11. 使用最优参数创建新模型
print("\n用最优参数训练最终模型...")
best_model = MultiOutputRegressor(MLPRegressor(**best_params))

# 12. 用原始训练数据训练最终模型
best_model.fit(X_train_scaled, y_train)
print("模型训练完成！")

# 13. 在测试集上进行预测
y_pred = best_model.predict(X_test_scaled)

# 14. 评估最终模型性能
print("\n优化后模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0
min_r2 = float('inf')
min_r2_var = ''

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 找出最小R²的变量
    if r2 < min_r2:
        min_r2 = r2
        min_r2_var = name
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

# 15. 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")
print(f"最小 R²: {min_r2:.4f} (变量: {min_r2_var})")

# 16. 输出贝叶斯优化的最佳结果
print("\n贝叶斯优化最佳结果:")
print(f"最佳最小R²: {-result.fun:.4f}")
print(f"优化迭代次数: {len(result.func_vals)}")

# 17. 为比较创建一个基准模型（使用Mlp.ipynb中的默认参数作为基准）
print("\n训练对照模型...")
baseline_params = {
    'hidden_layer_sizes': (100, 50),
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.0001,
    'batch_size': 'auto',
    'learning_rate': 'adaptive',
    'learning_rate_init': 0.001,
    'max_iter': 1000,
    'tol': 0.0001,
    'early_stopping': True,
    'validation_fraction': 0.1,
    'n_iter_no_change': 10,
    'random_state': 42
}

baseline_model = MultiOutputRegressor(MLPRegressor(**baseline_params))

# 用原始训练数据训练基准模型
baseline_model.fit(X_train_scaled, y_train)

# 在测试集上评估基准模型
y_pred_baseline = baseline_model.predict(X_test_scaled)

# 计算基准模型指标
baseline_r2_vals = []
baseline_r2 = 0
baseline_rmse = 0
baseline_min_r2 = float('inf')
baseline_min_r2_var = ''

print("\n基准模型各变量R²:")
for idx, name in enumerate(target_vars):
    r2 = r2_score(y_test.iloc[:, idx], y_pred_baseline[:, idx])
    baseline_r2_vals.append(r2)
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred_baseline[:, idx])
    rmse = np.sqrt(mse)
    
    print(f"{name}: {r2:.4f}")
    
    if r2 < baseline_min_r2:
        baseline_min_r2 = r2
        baseline_min_r2_var = name
        
    baseline_r2 += r2
    baseline_rmse += rmse

baseline_r2 /= len(target_vars)
baseline_rmse /= len(target_vars)

print(f"\n基准模型平均R²: {baseline_r2:.4f}")
print(f"基准模型最小R²: {baseline_min_r2:.4f} (变量: {baseline_min_r2_var})")
print(f"基准模型平均RMSE: {baseline_rmse:.4f}")
print(f"优化模型平均R²: {overall_r2:.4f}")
print(f"优化模型最小R²: {min_r2:.4f} (变量: {min_r2_var})")
print(f"优化模型平均RMSE: {overall_rmse:.4f}")
print(f"平均R²改进: {(overall_r2 - baseline_r2):.4f}")
print(f"最小R²改进: {(min_r2 - baseline_min_r2):.4f}")
print(f"RMSE改进: {(baseline_rmse - overall_rmse):.4f}")